# EarningsLens — Stage 7: Sentiment Analysis

Stage 7 computes sentiment scores across the 2023–2025 Specific FLS sentence corpus using two complementary models:

**FinBERT** (`ProsusAI/finbert`) — a BERT model fine-tuned on financial text. Outputs `positive`, `negative`, or `neutral` with a softmax probability per sentence. FinBERT captures domain-specific financial sentiment that general-purpose models miss — words like "headwinds", "challenging", "robust", and "normalisation" carry financial-specific sentiment that FinBERT handles correctly.

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) — a lexicon and rule-based sentiment analyser optimised for short texts. Outputs a compound score in [-1, 1]. VADER is fast (CPU, no GPU required), handles punctuation and capitalisation as sentiment signals, and complements FinBERT by capturing surface-level tonal cues.

### Why sentiment on FLS sentences specifically

Scoring sentiment on Specific FLS sentences rather than full transcripts gives a more targeted signal — it measures the tone with which management delivers its commitments, not the overall call tone. A transcript can be generally positive while specific guidance is delivered cautiously, or vice versa. This distinction is meaningful for the credibility framework.

### Output

Sentence-level scores are written back to `fls_2023_2025.parquet`. Transcript-level aggregates (mean FinBERT score, mean VADER compound, dominant label) are written to `sentiment_scores.parquet` for direct use by the Stage 10 dashboard and the Stage 9 agentic layer.

## 0. Dependencies

`vaderSentiment` is the VADER library — CPU-only, fast, no model download required. `ProsusAI/finbert` is downloaded from HuggingFace on first run. Both `transformers` and `torch` are already installed from earlier stages.

In [ ]:
# %pip install -q vaderSentiment transformers torch tqdm pandas

## 0.1 Imports & Device Configuration

In [1]:
# Standard library
import os
import logging
from pathlib import Path
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

# Deep learning
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Data
import pandas as pd
import numpy as np
from tqdm import tqdm

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens.stage7")

if not torch.cuda.is_available():
    log.warning("CUDA not available — FinBERT will run on CPU (~2-3 hrs)")
    DEVICE = torch.device("cpu")
else:
    DEVICE = torch.device("cuda")
    log.info("CUDA device : %s", torch.cuda.get_device_name(0))


load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise EnvironmentError("DATABASE_URL not found — check your .env file")

print("✓ Imports OK")
print(f"  Device : {DEVICE}")

c:\Users\sidsu\Desktop\Sem 4\NLP\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 03:54:20,438 [INFO] CUDA device : NVIDIA GeForce RTX 4060 Laptop GPU


✓ Imports OK
  Device : cuda


## 1. Configuration

`FINBERT_MODEL` is `ProsusAI/finbert` — distinct from `finbert-tone` used in Stage 5. ProsusAI/finbert is fine-tuned on financial phrasebank data and outputs `positive`, `negative`, `neutral` labels. It has proper tokenizer files on HuggingFace and loads without the workarounds needed for `finbert-tone`.

In [2]:
# Paths
INPUT_PATH     = Path("fls_2023_2025.parquet")
SENTIMENT_PATH = Path("sentiment_scores.parquet")  # transcript-level aggregates

# FinBERT
FINBERT_MODEL  = "ProsusAI/finbert"
BATCH_SIZE     = 64
MAX_LENGTH     = 128

print("✓ Configuration set")
print(f"  FinBERT model  : {FINBERT_MODEL}")
print(f"  Batch size     : {BATCH_SIZE}")

✓ Configuration set
  FinBERT model  : ProsusAI/finbert
  Batch size     : 64


## 2. Load Data

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH} not found. "
        "Ensure Stage 6 completed and saved fls_2023_2025.parquet."
    )

In [4]:
fls = pd.read_parquet(INPUT_PATH)
log.info("Loaded %d rows from %s", len(fls), INPUT_PATH)

print(f"Total sentences      : {len(fls):,}")
print(f"Unique transcripts   : {fls['transcript_id'].nunique():,}")
print(f"Unique tickers       : {fls['ticker'].nunique():,}")
print(f"Columns              : {list(fls.columns)}")

2026-05-07 03:54:20,693 [INFO] Loaded 92458 rows from fls_2023_2025.parquet


Total sentences      : 92,458
Unique transcripts   : 4,664
Unique tickers       : 531
Columns              : ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence', 'fls_label', 'fls_confidence', 'sentence_hash', 'metric', 'value', 'timeframe', 'lm_score', 'tone_label', 'tone_confidence', 'tone_score', 'hedge_score', 'commitment_id', 'status', 'matched_commitment_id', 'slot_count', 'quarter_key', 'metric_norm', 'match_similarity', 'vader_compound', 'finbert_label', 'finbert_confidence', 'finbert_score']


## 3. VADER Sentiment — Signal 1

VADER runs on CPU and processes 269k sentences in under two minutes. The compound score is the primary VADER output — a float in [-1, 1] where values above 0.05 are conventionally positive, below -0.05 negative, and between them neutral. The compound score is stored directly as `vader_compound`.

In [5]:
vader = SentimentIntensityAnalyzer()

In [6]:

log.info("Running VADER over %d sentences...", len(fls))

# VADER is fast enough to run without batching — apply directly
fls["vader_compound"] = fls["sentence"].apply(
    lambda s: vader.polarity_scores(str(s))["compound"]
)

log.info("VADER complete")

2026-05-07 03:54:20,729 [INFO] Running VADER over 92458 sentences...
2026-05-07 03:54:25,803 [INFO] VADER complete


In [7]:
print("VADER compound score distribution:")
print(fls["vader_compound"].describe().round(3).to_string())

# Label distribution using standard VADER thresholds
vader_pos = (fls["vader_compound"] >  0.05).sum()
vader_neg = (fls["vader_compound"] < -0.05).sum()
vader_neu = len(fls) - vader_pos - vader_neg

print(f"\nVADER label distribution (standard thresholds):")
print(f"  Positive (> 0.05)    : {vader_pos:>8,}  ({vader_pos/len(fls)*100:.1f}%)")
print(f"  Neutral  (-0.05–0.05): {vader_neu:>8,}  ({vader_neu/len(fls)*100:.1f}%)")
print(f"  Negative (< -0.05)   : {vader_neg:>8,}  ({vader_neg/len(fls)*100:.1f}%)")

VADER compound score distribution:
count    92458.000
mean         0.307
std          0.356
min         -0.956
25%          0.000
50%          0.326
75%          0.612
max          0.995

VADER label distribution (standard thresholds):
  Positive (> 0.05)    :   57,012  (61.7%)
  Neutral  (-0.05–0.05):   28,291  (30.6%)
  Negative (< -0.05)   :    7,155  (7.7%)


## 4. Load ProsusAI/finbert

`ProsusAI/finbert` has proper tokenizer files on HuggingFace and loads cleanly via `AutoTokenizer` — no workarounds needed unlike `finbert-tone` in Stage 5.

In [8]:
log.info("Loading %s...", FINBERT_MODEL)

finbert_tokeniser = AutoTokenizer.from_pretrained(FINBERT_MODEL)
finbert_model     = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_MODEL,
    torch_dtype = torch.float16,
)
finbert_model.to(DEVICE)
finbert_model.eval()

log.info("Label mapping : %s", finbert_model.config.id2label)
print("✓ ProsusAI/finbert loaded")
print(f"  Labels : {finbert_model.config.id2label}")

2026-05-07 03:54:25,831 [INFO] Loading ProsusAI/finbert...
2026-05-07 03:54:26,311 [INFO] HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 03:54:26,336 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-05-07 03:54:26,574 [INFO] HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 03:54:26,591 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-07 03:54:26,839 [INFO] HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-07 03:54:27,075 [INFO] HTTP Request: GET https://hugg

✓ ProsusAI/finbert loaded
  Labels : {0: 'positive', 1: 'negative', 2: 'neutral'}


2026-05-07 03:54:29,527 [INFO] HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-07 03:54:29,764 [INFO] HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"


## 5. FinBERT Inference — Signal 2

FinBERT runs over all 269k sentences in batches of 64. The predicted label and its softmax confidence are stored per sentence. A numeric score is also derived — mapping `positive` → +confidence, `neutral` → 0, `negative` → -confidence — giving a continuous [-1, 1] scale consistent with the VADER compound score for cross-signal comparison.

In [9]:
def run_finbert_batch(texts: list[str]) -> tuple[list[str], list[float], list[float]]:
    """
    Run one batch through ProsusAI/finbert.

    Returns:
        labels      : Predicted sentiment label per sentence
        confidences : Softmax probability of predicted label
        scores      : Numeric score in [-1, 1] derived from label + confidence
    """
    encoded = finbert_tokeniser(
        texts,
        padding        = True,
        truncation     = True,
        max_length     = MAX_LENGTH,
        return_tensors = "pt",
    ).to(DEVICE)

    with torch.inference_mode():
        logits = finbert_model(**encoded).logits

    probs      = torch.softmax(logits, dim=-1).cpu()
    pred_ids   = probs.argmax(dim=-1).tolist()
    pred_probs = probs.max(dim=-1).values.tolist()
    labels     = [finbert_model.config.id2label[i] for i in pred_ids]

    # Numeric score: positive → +conf, neutral → 0, negative → -conf
    scores = []
    for label, conf in zip(labels, pred_probs):
        l = label.lower()
        if l == "positive":
            scores.append(conf)
        elif l == "negative":
            scores.append(-conf)
        else:
            scores.append(0.0)

    return labels, pred_probs, scores

In [10]:
sentences_list     = fls["sentence"].tolist()
all_fb_labels      = []
all_fb_confidences = []
all_fb_scores      = []

In [11]:
log.info(
    "Running FinBERT over %d sentences — ~20–30 min on RTX 4060...",
    len(sentences_list)
)

for batch_start in tqdm(range(0, len(sentences_list), BATCH_SIZE), desc="FinBERT"):
    batch = sentences_list[batch_start : batch_start + BATCH_SIZE]
    labels, confs, scores = run_finbert_batch(batch)
    all_fb_labels.extend(labels)
    all_fb_confidences.extend(confs)
    all_fb_scores.extend(scores)

fls["finbert_label"]      = all_fb_labels
fls["finbert_confidence"] = all_fb_confidences
fls["finbert_score"]      = all_fb_scores

log.info("FinBERT inference complete")
print("\nFinBERT label distribution:")
print(fls["finbert_label"].value_counts().to_string())
print(f"\nFinBERT score distribution:")
print(fls["finbert_score"].describe().round(3).to_string())

2026-05-07 03:54:29,485 [INFO] Running FinBERT over 92458 sentences — ~20–30 min on RTX 4060...
FinBERT: 100%|██████████| 1445/1445 [01:08<00:00, 21.19it/s]
2026-05-07 03:55:37,751 [INFO] FinBERT inference complete



FinBERT label distribution:
finbert_label
positive    59571
neutral     23205
negative     9682

FinBERT score distribution:
count    92458.000
mean         0.467
std          0.596
min         -0.977
25%          0.000
50%          0.811
75%          0.937
max          0.963


## 6. Validation — Sample Sentences by Sentiment

Qualitative check of the FinBERT and VADER scores — the highest positive and most negative sentences are printed to confirm both models are responding to financial language correctly.

In [12]:
print("── Most positive FinBERT sentences ──")
top_pos = fls.nlargest(5, "finbert_score")[["sentence", "finbert_label", "finbert_score", "vader_compound"]]
for _, row in top_pos.iterrows():
    print(f"  [finbert={row['finbert_score']:.3f} | vader={row['vader_compound']:.3f}]")
    print(f"  {str(row['sentence'])[:120]}")
    print()

── Most positive FinBERT sentences ──
  [finbert=0.963 | vader=0.656]
  In the second half of 2023, we now expect year-over-year growth in programming cost per video customer to be similar to 

  [finbert=0.963 | vader=0.459]
  Jim, we are expecting margin improvement in every segment.

  [finbert=0.962 | vader=0.527]
  In Energy & Transportation, we anticipate slightly higher sales versus the prior year, supported by power generation.

  [finbert=0.962 | vader=0.202]
  We anticipate lower contribution from branded pharmaceuticals, which includes lower volumes for HUMIRA, and we anticipat

  [finbert=0.961 | vader=0.700]
  No, we are expecting to see nice improvement in international margins.



In [13]:
print("── Most negative FinBERT sentences ──")
top_neg = fls.nsmallest(5, "finbert_score")[["sentence", "finbert_label", "finbert_score", "vader_compound"]]
for _, row in top_neg.iterrows():
    print(f"  [finbert={row['finbert_score']:.3f} | vader={row['vader_compound']:.3f}]")
    print(f"  {str(row['sentence'])[:120]}")
    print()

── Most negative FinBERT sentences ──
  [finbert=-0.977 | vader=0.586]
  Constant dollar revenue growth at the midpoint is almost 3%, adjusted EPS in 1Q '25 is expected to be $1.33 to $1.43 per

  [finbert=-0.977 | vader=0.000]
  First quarter revenue is expected to be in the range of $4.6 billion to $4.8 billion, down 3% at the midpoint.

  [finbert=-0.977 | vader=0.000]
  In the medium-duty truck market, we expect the market size to be 140,000 to 155,000 units, down 5% to 15% compared to 20

  [finbert=-0.977 | vader=0.296]
  Operating EPS is now expected to be in the range of $2.50 to $2.70 per share, down 3% versus prior year at the midpoint.

  [finbert=-0.977 | vader=0.000]
  UGG revenue is still expected to be down mid-single-digits on a reported basis, implying a year-over-year decline in the



In [14]:
# Signal agreement — how often do FinBERT and VADER agree on sign
finbert_sign = np.sign(fls["finbert_score"])
vader_sign   = np.sign(fls["vader_compound"])
agreement    = (finbert_sign == vader_sign).mean() * 100
print(f"FinBERT / VADER sign agreement : {agreement:.1f}%")

FinBERT / VADER sign agreement : 64.6%


## 7. Transcript-Level Sentiment Aggregates

Sentence-level scores are aggregated per transcript and per segment type (`prepared_remarks` vs `qa`). The dashboard displays these aggregate scores — not per-sentence scores — as the supporting tone context alongside the guidance delta table.

Aggregation uses mean scores rather than majority voting to preserve gradient information. A transcript where half the sentences are strongly positive and half are strongly negative has a mean near zero, which is more informative than a neutral majority label.

In [15]:
# Aggregate per transcript × segment type
sentiment_agg = (
    fls.groupby(["transcript_id", "ticker", "year", "quarter", "segment_type"])
    .agg(
        finbert_mean    = ("finbert_score",  "mean"),
        finbert_pos_pct = ("finbert_label",  lambda x: (x.str.lower() == "positive").mean()),
        finbert_neg_pct = ("finbert_label",  lambda x: (x.str.lower() == "negative").mean()),
        vader_mean      = ("vader_compound", "mean"),
        sentence_count  = ("sentence",       "count"),
    )
    .reset_index()
)

In [16]:
# Dominant label per transcript-segment
def dominant_label(finbert_mean: float) -> str:
    if finbert_mean > 0.1:
        return "positive"
    elif finbert_mean < -0.1:
        return "negative"
    return "neutral"

In [17]:
sentiment_agg["dominant_label"] = sentiment_agg["finbert_mean"].apply(dominant_label)

print(f"Transcript-segment records : {len(sentiment_agg):,}")
print(f"\nDominant label distribution:")
print(sentiment_agg["dominant_label"].value_counts().to_string())
print(f"\nSample aggregate records:")
print(sentiment_agg.head(10).to_string(index=False))

Transcript-segment records : 8,430

Dominant label distribution:
dominant_label
positive    7494
neutral      741
negative     195

Sample aggregate records:
transcript_id ticker  year  quarter     segment_type  finbert_mean  finbert_pos_pct  finbert_neg_pct  vader_mean  sentence_count dominant_label
  AAL_2023_Q1    AAL  2023        1 prepared_remarks      0.199444         0.538462         0.230769    0.116585              13       positive
  AAL_2023_Q1    AAL  2023        1               qa      0.432454         0.500000         0.000000    0.122650              12       positive
  AAL_2023_Q2    AAL  2023        2 prepared_remarks      0.377991         0.625000         0.125000    0.052006              16       positive
  AAL_2023_Q2    AAL  2023        2               qa      0.352246         0.400000         0.000000    0.337520               5       positive
  AAL_2023_Q3    AAL  2023        3 prepared_remarks      0.533514         0.727273         0.136364    0.165423          

## 8. Write Results - Local Parquet and Supabase

Sentence-level scores are written back to `fls_2023_2025.parquet`. Transcript-level aggregates are written to `sentiment_scores.parquet` and `Supabase` — the file Stage 9 (agentic layer) and Stage 10 (dashboard) read directly.

In [18]:
# Write sentence-level scores back to main cache
fls.to_parquet(INPUT_PATH, index=False)
log.info("fls_2023_2025.parquet updated with sentiment scores")

2026-05-07 03:55:42,418 [INFO] fls_2023_2025.parquet updated with sentiment scores


In [19]:
# Write transcript-level aggregates
sentiment_agg.to_parquet(SENTIMENT_PATH, index=False)
log.info("sentiment_scores.parquet saved")

print("✓ Outputs written")
print(f"  fls_2023_2025.parquet    : {len(fls):,} rows")
print(f"  sentiment_scores.parquet : {len(sentiment_agg):,} rows")
print(f"  Columns in fls           : {list(fls.columns)}")

2026-05-07 03:55:42,453 [INFO] sentiment_scores.parquet saved


✓ Outputs written
  fls_2023_2025.parquet    : 92,458 rows
  sentiment_scores.parquet : 8,430 rows
  Columns in fls           : ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence', 'fls_label', 'fls_confidence', 'sentence_hash', 'metric', 'value', 'timeframe', 'lm_score', 'tone_label', 'tone_confidence', 'tone_score', 'hedge_score', 'commitment_id', 'status', 'matched_commitment_id', 'slot_count', 'quarter_key', 'metric_norm', 'match_similarity', 'vader_compound', 'finbert_label', 'finbert_confidence', 'finbert_score']


In [20]:
sentiment_agg = pd.read_parquet("sentiment_scores.parquet")

In [21]:
sentiment_agg.columns

Index(['transcript_id', 'ticker', 'year', 'quarter', 'segment_type',
       'finbert_mean', 'finbert_pos_pct', 'finbert_neg_pct', 'vader_mean',
       'sentence_count', 'dominant_label'],
      dtype='str')

In [22]:
conn   = psycopg2.connect(DATABASE_URL + "?sslmode=require")
cursor = conn.cursor()

sentiment_rows = [
    (row["transcript_id"], row["segment_type"], row["finbert_mean"], row["vader_mean"])
    for _, row in sentiment_agg.iterrows()
]

execute_values(
    cursor,
    """
    INSERT INTO sentiment_scores (transcript_id, segment_type, finbert_score, vader_score)
    VALUES %s
    ON CONFLICT DO NOTHING
    """,
    sentiment_rows,
    page_size=2000
)
conn.commit()
conn.close()
print(f"✓ Stage 7 — {len(sentiment_rows):,} sentiment rows written")

✓ Stage 7 — 8,430 sentiment rows written


In [ ]:
print(sentiment_agg.groupby("segment_type").size())
# prepared_remarks: ~4,400
# qa: ~4,030
# (or similar — they won't be equal)

# Cross-check: should equal 8,430
print(sentiment_agg[["transcript_id", "segment_type"]].drop_duplicates().shape[0])

segment_type
prepared_remarks    4342
qa                  4088
dtype: int64
8430


: 

## Stage 7 Summary

In [23]:
print("=" * 52)
print("  EarningsLens — Stage 7 Complete")
print("=" * 52)
print(f"\nSentences scored         : {len(fls):,}")
print(f"Unique transcripts       : {fls['transcript_id'].nunique():,}")

print(f"\nFinBERT:")
for label, count in fls["finbert_label"].value_counts().items():
    print(f"  {label:<12}: {count:>8,}  ({count/len(fls)*100:.1f}%)")
print(f"  Mean score     : {fls['finbert_score'].mean():.3f}")

print(f"\nVADER:")
print(f"  Mean compound  : {fls['vader_compound'].mean():.3f}")
print(f"  Std            : {fls['vader_compound'].std():.3f}")

print(f"\nOutputs:")
print(f"  fls_2023_2025.parquet    — sentence-level scores added")
print(f"  sentiment_scores.parquet — {len(sentiment_agg):,} transcript-segment aggregates")
print(f"\n→ Ready for Stage 8 — LLM Summarisation & RAG")

  EarningsLens — Stage 7 Complete

Sentences scored         : 92,458
Unique transcripts       : 4,664

FinBERT:
  positive    :   59,571  (64.4%)
  neutral     :   23,205  (25.1%)
  negative    :    9,682  (10.5%)
  Mean score     : 0.467

VADER:
  Mean compound  : 0.307
  Std            : 0.356

Outputs:
  fls_2023_2025.parquet    — sentence-level scores added
  sentiment_scores.parquet — 8,430 transcript-segment aggregates

→ Ready for Stage 8 — LLM Summarisation & RAG


## Handoff to Stage 8

Stage 7 is complete. The outputs are:

| Output | Location | Consumed by |
|---|---|---|
| `fls_2023_2025.parquet` | Local disk | Stage 8, 9, 10 |
| `sentiment_scores.parquet` | Local disk | Stage 9 (agentic), Stage 10 (dashboard) |

`fls_2023_2025.parquet` now carries the complete pipeline output for all 269,679 Specific FLS sentences: FLS label, confidence, metric, value, timeframe, hedge score, cross-quarter status, match similarity, FinBERT score, and VADER compound.

Stage 8 builds the LLM summarisation and RAG pipeline. Ollama generates plain-English commitment summaries per company grounded in the structured pipeline outputs. LangChain and ChromaDB power the natural language Q&A interface that allows users to ask questions about any company's guidance history across quarters.